In [2]:
# ==========================================================
# Notebook 08: Enterprise Search API
# ==========================================================

import ast
import re
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder

from fastapi import FastAPI, HTTPException

from pydantic import BaseModel

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df["acl_roles"] = enterprise_df["acl_roles"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

enterprise_df["search_text"] = enterprise_df["title"] + " " + enterprise_df["content"]

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team,search_text
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","[Engineering, Public]",Internal,Engineering,Phoenix Deployment Discussion The Phoenix-2026...
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","[Finance, Public]",Restricted,Finance,Model Training Update The semantic search mode...
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","[HR, Public]",Restricted,HR,Phoenix Architecture Wiki Project Phoenix-2026...
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","[Engineering, Public]",Internal,Engineering,Enterprise Search Design Hybrid search combine...
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","[Engineering, Public]",Internal,Engineering,PHX-245 Database Migration Complete PostgreSQL...


In [4]:
USER_PROFILES = {
    "alice": {"roles": ["Engineering", "Public"]},
    "bob": {"roles": ["HR", "Public"]},
    "carol": {"roles": ["Finance", "Public"]},
    "david": {"roles": ["Legal", "Public"]},
}

In [5]:
def acl_filter(dataframe, user_roles):

    mask = dataframe["acl_roles"].apply(
        lambda acl: any(role in acl for role in user_roles)
    )

    return dataframe[mask].reset_index(drop=True)

In [6]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)

    return text.split()


tokenized_corpus = [preprocess_text(doc) for doc in enterprise_df["search_text"]]

bm25_model = BM25Okapi(tokenized_corpus)

In [7]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
document_embeddings = embedding_model.encode(
    enterprise_df["search_text"].tolist(), convert_to_numpy=True, show_progress_bar=True
)

document_embeddings = np.array(document_embeddings).astype("float32")

faiss.normalize_L2(document_embeddings)

faiss_index = faiss.IndexFlatIP(document_embeddings.shape[1])

faiss_index.add(document_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
def retrieve_candidates(query, top_k=20):

    bm25_scores = bm25_model.get_scores(preprocess_text(query))

    bm25_rank = np.argsort(bm25_scores)[::-1][:top_k]

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    _, semantic_rank = faiss_index.search(query_embedding, top_k)

    candidates = list(set(list(bm25_rank) + list(semantic_rank[0])))

    return candidates

In [10]:
def rerank_documents(query, candidate_indices, dataframe, top_k=5):

    pairs = [(query, dataframe.iloc[idx]["search_text"]) for idx in candidate_indices]

    scores = cross_encoder.predict(pairs)

    output = []

    for idx, score in zip(candidate_indices, scores):

        row = dataframe.iloc[idx].to_dict()

        row["score"] = float(score)

        output.append(row)

    output = sorted(output, key=lambda x: x["score"], reverse=True)

    return output[:top_k]

In [11]:
class SearchRequest(BaseModel):

    user_name: str

    query: str

    top_k: int = 5


class SearchResult(BaseModel):

    doc_id: str

    title: str

    department: str

    score: float

In [12]:
app = FastAPI(
    title="Enterprise Search API",
    description=("Secure Hybrid Enterprise " "Search Engine"),
    version="1.0.0",
)

In [13]:
@app.get("/health")
def health_check():

    return {"status": "healthy", "service": "enterprise-search-api"}

In [15]:
!GET http://127.0.0.1:8000/health

'GET' is not recognized as an internal or external command,
operable program or batch file.


In [16]:
@app.post("/search")
def search_documents(request: SearchRequest):

    if request.user_name not in USER_PROFILES:

        raise HTTPException(status_code=404, detail="Unknown user.")

    user_roles = USER_PROFILES[request.user_name]["roles"]

    authorized_df = acl_filter(enterprise_df, user_roles)

    authorized_df = authorized_df.reset_index(drop=True)

    candidate_docs = retrieve_candidates(request.query, top_k=20)

    candidate_docs = [idx for idx in candidate_docs if idx < len(authorized_df)]

    results = rerank_documents(
        request.query, candidate_docs, authorized_df, top_k=request.top_k
    )

    return {"user": request.user_name, "query": request.query, "results": results}

In [17]:
sample_request = SearchRequest(
    user_name="alice", query="vector database migration", top_k=3
)

response = search_documents(sample_request)

response

{'user': 'alice',
 'query': 'vector database migration',
 'results': [{'source': 'Jira',
   'title': 'PHX-245 Database Migration',
   'content': 'Complete PostgreSQL migration before Phoenix production launch.',
   'author': 'Eric',
   'doc_id': 'DOC0005',
   'department': 'Engineering',
   'created_date': '2026-01-15',
   'tags': "['enterprise-search', 'kubernetes']",
   'acl_roles': ['Engineering', 'Public'],
   'security_level': 'Internal',
   'owner_team': 'Engineering',
   'search_text': 'PHX-245 Database Migration Complete PostgreSQL migration before Phoenix production launch.',
   'score': -1.2734522819519043},
  {'source': 'Slack',
   'title': 'Phoenix Deployment Discussion',
   'content': 'The Phoenix-2026 deployment is scheduled for next Friday. Database migration will happen during the maintenance window.',
   'author': 'Alice',
   'doc_id': 'DOC0001',
   'department': 'Engineering',
   'created_date': '2026-01-11',
   'tags': "['enterprise-search', 'semantic-search']",
   '